# DRAFT: End-to-End Colab Training, Export, and Evaluation

This notebook runs the complete **DRAFT** pipeline in one pass (`Runtime -> Run all`) and backs up every trained checkpoint, index, GGUF export, and evaluation report to Google Drive after each stage completes.

### Before Running
1. Upload the project folder (`configs/`, `prompts/`, `scripts/`, `src/`, `data/org_docs/`, `pyproject.toml`, and optionally `data/processed/`) to `MyDrive/draft-rag/`, **or** upload `team-project-src.zip` inside `MyDrive/draft-rag/`.
2. Select a GPU runtime: **Runtime -> Change runtime type -> Hardware accelerator: GPU** (A100 40GB recommended; L4 and T4 are also supported automatically).
3. Run **Runtime -> Run all**.

### Outputs Saved to Google Drive (`MyDrive/draft-rag/`)
- `data/processed/`: canonical `train.parquet`, `val.parquet`, `test.parquet`, and `sft/` JSONL datasets
- `artifacts/classifier_baseline/` and `artifacts/classifier/`: Stage 1 TF-IDF baseline and fine-tuned `microsoft/deberta-v3-small`
- `artifacts/retriever/`: Stage 2 fine-tuned `BAAI/bge-small-en-v1.5` encoder and rebuilt dual FAISS + BM25 index
- `artifacts/llm/`: Stage 3 `Qwen2.5-3B-Instruct` LoRA adapter, `Modelfile`, and `draft-qwen25-3b-q8_0.gguf` for local Ollama
- `reports/`: `data_summary.json`, `classifier_baseline_metrics.json`, `classifier_transformer_metrics.json`, `retriever_training_metrics.json`, `retrieval_metrics.json`, `llm_training_metrics.json`, and `ablation_table.json`


In [ ]:
# Editable Configuration
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/draft-rag"
DRIVE_ZIP_PATH = ""  # Optional explicit path to a source .zip on Drive
WORK_DIR = "/content/draft-rag"

SKIP_IF_DONE = True  # Reuse existing checkpoints restored from Drive if already trained

RUN_STAGE1 = True
RUN_STAGE2 = True
RUN_STAGE3 = True
RUN_GGUF_EXPORT = True
RUN_ABLATION = True

SFT_TICKET_SAMPLES = 3000
ABLATION_SAMPLES = 100


In [ ]:
import subprocess
import torch
from google.colab import drive

drive.mount("/content/drive")

subprocess.run(["nvidia-smi"], check=True)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not detected. Enable GPU under Runtime -> Change runtime type.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
try:
    bf16_ok = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:
    bf16_ok = torch.cuda.is_bf16_supported()

if vram_gb >= 32 and bf16_ok:
    GPU_PROFILE = "A100/H100 BF16"
    CLS_BATCH = 64
    RET_BATCH = 128
    LLM_BATCH = 8
    LLM_GRAD_ACCUM = 2
    LLM_NO_4BIT = True
elif vram_gb >= 20 and bf16_ok:
    GPU_PROFILE = "L4 BF16 QLoRA"
    CLS_BATCH = 32
    RET_BATCH = 64
    LLM_BATCH = 4
    LLM_GRAD_ACCUM = 4
    LLM_NO_4BIT = False
else:
    GPU_PROFILE = "T4 QLoRA"
    CLS_BATCH = 32
    RET_BATCH = 64
    LLM_BATCH = 2
    LLM_GRAD_ACCUM = 8
    LLM_NO_4BIT = False

print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM, bf16={bf16_ok}) -> Profile: {GPU_PROFILE}")
print(
    f"Auto-selected batches: CLS={CLS_BATCH}, RET={RET_BATCH}, "
    f"LLM={LLM_BATCH}x{LLM_GRAD_ACCUM}, no_4bit={LLM_NO_4BIT}"
)


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

drive_dir = Path(DRIVE_PROJECT_DIR)
work_dir = Path(WORK_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)

if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

zip_candidate = Path(DRIVE_ZIP_PATH) if DRIVE_ZIP_PATH else (drive_dir / "team-project-src.zip")
if (drive_dir / "src").exists() and not DRIVE_ZIP_PATH:
    for item in ("configs", "prompts", "scripts", "src", "data", "pyproject.toml", "artifacts", "reports"):
        src_path = drive_dir / item
        if src_path.exists():
            dst_path = work_dir / item
            if src_path.is_dir():
                shutil.copytree(src_path, dst_path, dirs_exist_ok=True)
            else:
                shutil.copy2(src_path, dst_path)
elif zip_candidate.exists():
    with zipfile.ZipFile(zip_candidate, "r") as zf:
        zf.extractall(work_dir)
    if not (work_dir / "src").exists():
        nested = [p for p in work_dir.iterdir() if p.is_dir() and (p / "src").exists()]
        if len(nested) == 1:
            for child in nested[0].iterdir():
                shutil.move(str(child), str(work_dir / child.name))
            nested[0].rmdir()
    for resumable in ("data/processed", "artifacts", "reports"):
        src_res = drive_dir / resumable
        if src_res.exists():
            shutil.copytree(src_res, work_dir / resumable, dirs_exist_ok=True)
else:
    raise FileNotFoundError(
        f"Could not find source files in {drive_dir} or zip archive at {zip_candidate}."
    )

os.chdir(work_dir)
os.environ["PYTHONUNBUFFERED"] = "1"
print("Working directory:", Path.cwd())


def run_step(cmd: list[str]) -> None:
    '''Run a subprocess command with live streaming and raise immediately on failure.'''
    print(">>>", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")


def sync_to_drive(rel_path: str) -> None:
    '''Incrementally copy a local file or directory to Google Drive.'''
    src = work_dir / rel_path
    dst = drive_dir / rel_path
    if not src.exists():
        print(f"Skip Drive sync (missing): {rel_path}")
        return
    if src.is_dir():
        shutil.copytree(
            src,
            dst,
            ignore=shutil.ignore_patterns("checkpoints", "__pycache__", "*.tmp"),
            dirs_exist_ok=True,
        )
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f"Synced to Drive: {dst}")


pkgs = [
    "trl", "peft", "bitsandbytes", "accelerate", "datasets", "transformers",
    "sentence-transformers", "safetensors", "sentencepiece", "protobuf",
    "pydantic", "pydantic-settings", "pyyaml", "jinja2", "typer", "httpx",
    "rank-bm25", "faiss-cpu", "rouge-score", "bert-score", "evaluate", "gguf",
]
run_step([sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs])
run_step([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."])


In [ ]:
import sys
from pathlib import Path

splits_ready = all(
    Path(f"data/processed/{name}.parquet").exists() for name in ("train", "val", "test")
)
if not splits_ready:
    run_step([sys.executable, "-m", "draft.cli", "prepare-data"])
    sync_to_drive("data/processed/train.parquet")
    sync_to_drive("data/processed/val.parquet")
    sync_to_drive("data/processed/test.parquet")
    sync_to_drive("reports/data_summary.json")
else:
    print("Canonical splits already present.")

baseline_done = Path("reports/classifier_baseline_metrics.json").exists()
if not (SKIP_IF_DONE and baseline_done):
    run_step([sys.executable, "-m", "draft.cli", "train-baseline"])
    sync_to_drive("artifacts/classifier_baseline")
    sync_to_drive("reports/classifier_baseline_metrics.json")
else:
    print("Skipping TF-IDF baseline (already present).")


In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd

cls_ckpt = Path("artifacts/classifier/model.safetensors")
if RUN_STAGE1:
    if SKIP_IF_DONE and cls_ckpt.exists():
        print("Skipping Stage 1 transformer training (checkpoint already exists).")
    else:
        run_step(
            [
                sys.executable,
                "scripts/colab/train_classifier.py",
                "--config",
                "configs/default.yaml",
                "--output-dir",
                "artifacts/classifier",
                "--batch-size",
                str(CLS_BATCH),
            ]
        )
        sync_to_drive("artifacts/classifier")
        sync_to_drive("reports/classifier_transformer_metrics.json")

rows = []
for name, path in (
    ("TF-IDF Baseline", Path("reports/classifier_baseline_metrics.json")),
    ("DeBERTa-v3-small", Path("reports/classifier_transformer_metrics.json")),
):
    if path.exists():
        data = json.loads(path.read_text(encoding="utf-8"))
        for split in ("val", "test"):
            m = data[split]
            rows.append(
                {
                    "Model": name,
                    "Split": split,
                    "Intent Acc": round(m["intent"]["accuracy"], 4),
                    "Intent Macro-F1": round(m["intent"]["macro_f1"], 4),
                    "Urgency Acc": round(m["urgency"]["accuracy"], 4),
                    "Urgency Macro-F1": round(m["urgency"]["macro_f1"], 4),
                    "Combined Macro-F1": round(m["combined_macro_f1"], 4),
                }
            )
if rows:
    display(pd.DataFrame(rows))


In [ ]:
import sys
from pathlib import Path

STAGE2_RAN = False
enc_dir = Path("artifacts/retriever/encoder")
idx_meta = Path("artifacts/retriever/index/index_meta.json")

if RUN_STAGE2:
    if SKIP_IF_DONE and enc_dir.exists() and idx_meta.exists():
        print("Skipping Stage 2 bi-encoder training (encoder and index already exist).")
    else:
        run_step(
            [
                sys.executable,
                "scripts/colab/train_retriever.py",
                "--config",
                "configs/default.yaml",
                "--output-dir",
                str(enc_dir),
                "--batch-size",
                str(RET_BATCH),
                "--epochs",
                "2",
                "--rebuild-index",
            ]
        )
        STAGE2_RAN = True
        sync_to_drive("artifacts/retriever")
        sync_to_drive("reports/retriever_training_metrics.json")

    if STAGE2_RAN or not Path("reports/retrieval_metrics.json").exists():
        run_step(
            [
                sys.executable,
                "-m",
                "draft.cli",
                "eval-retrieval",
                "--split",
                "test",
                "--max-queries",
                "300",
                "--rerank",
            ]
        )
        sync_to_drive("reports/retrieval_metrics.json")


In [ ]:
import sys
from pathlib import Path

sft_train = Path("data/processed/sft/sft_train.jsonl")
need_sft = (not SKIP_IF_DONE) or STAGE2_RAN or (not sft_train.exists())

if RUN_STAGE3 and need_sft:
    run_step(
        [
            sys.executable,
            "-m",
            "draft.cli",
            "build-sft-data",
            "--num-ticket-samples",
            str(SFT_TICKET_SAMPLES),
            "--no-rerank",
        ]
    )
    sync_to_drive("data/processed/sft")
else:
    print("Using existing SFT dataset.")


In [ ]:
import sys
from pathlib import Path

adapter_cfg = Path("artifacts/llm/lora_adapter/adapter_config.json")
if RUN_STAGE3:
    if SKIP_IF_DONE and adapter_cfg.exists():
        print("Skipping Stage 3 LLM fine-tuning (LoRA adapter already exists).")
    else:
        cmd = [
            sys.executable,
            "scripts/colab/train_llm.py",
            "--config",
            "configs/default.yaml",
            "--sft-dir",
            "data/processed/sft",
            "--output-dir",
            "artifacts/llm/lora_adapter",
            "--epochs",
            "2",
            "--batch-size",
            str(LLM_BATCH),
            "--grad-accum",
            str(LLM_GRAD_ACCUM),
            "--max-seq-length",
            "1536",
            "--merge-fp16",
        ]
        if LLM_NO_4BIT:
            cmd.append("--no-4bit")
        run_step(cmd)
        sync_to_drive("artifacts/llm/lora_adapter")
        sync_to_drive("artifacts/llm/Modelfile")
        sync_to_drive("reports/llm_training_metrics.json")


In [ ]:
import gc
import shutil
import sys
from pathlib import Path
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
from draft.config import load_config

cfg = load_config("configs/default.yaml")
gguf_path = Path(f"artifacts/llm/{cfg.generation.ollama_model}-q8_0.gguf")
merged_dir = Path("artifacts/llm/merged_fp16")
adapter_dir = Path("artifacts/llm/lora_adapter")

if RUN_GGUF_EXPORT:
    if SKIP_IF_DONE and gguf_path.exists():
        print("Skipping GGUF export (already exists).")
    else:
        llama_dir = Path("/content/llama.cpp")
        if not llama_dir.exists():
            run_step(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", str(llama_dir)])

        if not (merged_dir / "config.json").exists():
            if not adapter_dir.exists():
                raise FileNotFoundError(f"Missing LoRA adapter at {adapter_dir}")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            base = AutoModelForCausalLM.from_pretrained(
                cfg.generation.base_model,
                torch_dtype=torch.float16,
                device_map="cpu",
            )
            merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
            merged_dir.mkdir(parents=True, exist_ok=True)
            merged.save_pretrained(str(merged_dir), safe_serialization=True)
            AutoTokenizer.from_pretrained(str(adapter_dir)).save_pretrained(str(merged_dir))
            del base, merged
            gc.collect()

        run_step(
            [
                sys.executable,
                str(llama_dir / "convert_hf_to_gguf.py"),
                str(merged_dir),
                "--outfile",
                str(gguf_path),
                "--outtype",
                "q8_0",
            ]
        )
        sync_to_drive(str(gguf_path))
        sync_to_drive("artifacts/llm/Modelfile")

    if merged_dir.exists():
        shutil.rmtree(merged_dir, ignore_errors=True)
        print("Removed temporary local FP16 merge directory to reclaim disk space.")


In [ ]:
import sys
from pathlib import Path

ablation_report = Path("reports/ablation_table.json")
if RUN_ABLATION:
    if SKIP_IF_DONE and ablation_report.exists():
        print("Skipping 3-way ablation (reports/ablation_table.json already exists).")
    else:
        cls_backend = (
            "transformer"
            if Path("artifacts/classifier/model.safetensors").exists()
            else "baseline"
        )
        cmd = [
            sys.executable,
            "scripts/colab/eval_ablation.py",
            "--config",
            "configs/default.yaml",
            "--split",
            "test",
            "--max-samples",
            str(ABLATION_SAMPLES),
            "--classifier-backend",
            cls_backend,
            "--rerank",
            "--bertscore",
        ]
        if LLM_NO_4BIT:
            cmd.append("--no-4bit")
        run_step(cmd)
        sync_to_drive("reports/ablation_table.json")


In [ ]:
import json
from pathlib import Path
import pandas as pd

ret_report = Path("reports/retrieval_metrics.json")
if ret_report.exists():
    ret_data = json.loads(ret_report.read_text(encoding="utf-8"))
    print("Stage 2 Retrieval Ablations (Test):")
    org_rows = [
        {
            "Mode": mode,
            "Category Hit@1": round(m["category_hit@1"], 4),
            "Category Hit@3": round(m["category_hit@3"], 4),
            "Category MRR@3": round(m["category_mrr@3"], 4),
            "Unique Docs": m["unique_docs_retrieved"],
        }
        for mode, m in ret_data["org_docs"].items()
    ]
    display(pd.DataFrame(org_rows))

abl_report = Path("reports/ablation_table.json")
if abl_report.exists():
    abl = json.loads(abl_report.read_text(encoding="utf-8"))
    print(f"End-to-End 3-Way Generation Ablation (split={abl['split']}, n={abl['max_samples']}):")
    abl_rows = []
    for mode, data in abl["modes"].items():
        t = data["tickets"]
        p = data["policy_suite"]
        abl_rows.append(
            {
                "Mode": mode,
                "ROUGE-1": round(t["rouge"]["rouge1_f1"], 4),
                "ROUGE-L": round(t["rouge"]["rougeL_f1"], 4),
                "BERTScore F1": round(t.get("bertscore", {}).get("f1", 0.0), 4),
                "Citation Rate": round(t["policy_adherence"]["citation_rate"], 4),
                "Valid Cite Prec": round(t["policy_adherence"]["valid_citation_precision"], 4),
                "Policy Doc Hit": round(p["doc_retrieval_hit_rate"], 4),
                "Policy Fact Recall": round(p["policy_fact_recall"], 4),
                "Mean Latency (ms)": round(t["latency_ms"]["total_ms"]["mean"], 1),
            }
        )
    display(pd.DataFrame(abl_rows))

drive_root = Path(DRIVE_PROJECT_DIR)
print("\nSaved Drive Outputs:")
for folder in ("artifacts", "reports", "data/processed"):
    target = drive_root / folder
    if target.exists():
        files = sorted(p for p in target.rglob("*") if p.is_file())
        total_mb = sum(p.stat().st_size for p in files) / (1024 ** 2)
        print(f"  {target} -> {len(files)} files ({total_mb:.1f} MB)")
